# 每日脚本任务 重构

## 元数据
- 原始路径: g:\\我的云端硬盘\\workspace\\projects\\github\\work_station\\代码库\\daily\\每日脚本（最新）\\
- 创建时间: 2026-02-14
- 任务ID: T02
- 优先级: 最高

## 1. 项目概述

### 1.1 功能描述

每日脚本是债券研究的基础设施，包含定时数据获取、指标计算、报告生成等核心功能。

**主要功能模块**：
1. **数据获取脚本（7个）**：从THS API、交易所、中债等获取数据
2. **指标计算脚本（7个）**：计算行业指数、宏观指标、金融指标等
3. **行为分析脚本（4个）**：投资者情绪分析、资金流向分析等
4. **工具脚本（2个）**：测试和验证工具

### 1.2 数据源
- **THS API**：同花顺数据接口
- **交易所数据**：上交所、深交所、上清所
- **中债数据**：中国债券信息网
- **第三方API**：RatingDog等

### 1.3 输出结果
- 各类市场数据更新到数据库
- 指标计算结果
- 舆情监控报告
- 资金流向分析报告

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import json
import re
import random
import shutil
import string
from time import sleep
from datetime import datetime, date, timedelta
from urllib import parse
from io import BytesIO

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import pymysql
import sqlalchemy
from sqlalchemy import text, inspect
from sqlalchemy.pool import NullPool

# 网络请求
import requests
from bs4 import BeautifulSoup

# 进度条
from tqdm import tqdm
from multiprocessing import Pool

# 模糊匹配
from fuzzywuzzy import fuzz

# 配置导入
from config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR,
    DB_HOST, DB_PORT, DB_USER, DB_PASSWORD, DB_NAME,
    THS_USERNAME, THS_PASSWORD
)

print("依赖导入成功")

### 2.2 配置参数

In [ ]:
# 数据库连接字符串
DB_CONNECTION_STRING = f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

# 创建数据库引擎
sql_engine = sqlalchemy.create_engine(
    DB_CONNECTION_STRING,
    poolclass=NullPool
)

# 测试连接
try:
    with sql_engine.connect() as conn:
        print("数据库连接成功")
except Exception as e:
    print(f"数据库连接失败: {e}")

# 日期配置
today = datetime.now()
yesterday = today - timedelta(days=1)
today_str = today.strftime("%Y-%m-%d")
yesterday_str = yesterday.strftime("%Y-%m-%d")

print(f"今日日期: {today_str}")
print(f"昨日日期: {yesterday_str}")

## 3. 数据获取

### 3.1 数据源连接

In [ ]:
# THS API 登录
def login_ths(username, password):
    """登录同花顺API"""
    try:
        from iFinDPy import THS_iFinDLogin
        result = THS_iFinDLogin(username, password)
        if result == 0:
            print("THS API 登录成功")
            return True
        else:
            print(f"THS API 登录失败，错误码: {result}")
            return False
    except ImportError:
        print("警告: iFinDPy 模块未安装，THS API 功能不可用")
        return False

# 登录THS（如果配置了凭据）
if THS_USERNAME and THS_PASSWORD:
    ths_logged_in = login_ths(THS_USERNAME, THS_PASSWORD)

### 3.2 数据读取

从交易所API获取数据的示例函数：

In [ ]:
def process_df(df):
    """
    对DataFrame中的每一列进行处理，尝试将包含逗号的字符串转换为浮点数。
    
    参数:
    - df: pandas DataFrame
    
    返回:
    - 修改后的DataFrame
    """
    for column in df.columns:
        try:
            df[column] = df[column].str.replace(',', '')
            df[column] = pd.to_numeric(df[column], errors='raise')
        except:
            pass
    df = df.fillna(0)
    return df

def fetch_sse_data(year, month):
    """获取上交所数据"""
    trade_date = f'{year}-{month}'
    url = f"http://query.sse.com.cn/commonQuery.do?jsonCallBack=jsonpCallback73603&isPagination=false&sqlId=COMMON_BOND_SCSJ_SCTJ_TJYB_CYJG_L&TRADEDATE={trade_date}&_=1715595737219"
    
    headers = {
        'Accept': '*/*',
        'Referer': 'http://bond.sse.com.cn/',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.post(url, headers=headers, timeout=20)
        df = pd.DataFrame(json.loads(response.content.decode('utf-8')[len('jsonpCallback73603('):-1])['result'])
        df = process_df(df)
        return df
    except Exception as e:
        print(f"获取上交所数据失败: {e}")
        return None

print("数据获取函数定义完成")

## 4. 数据处理

### 4.1 数据清洗

In [ ]:
def generate_column_mapping(columns):
    """生成列名映射"""
    return {col: f"col_{i}" for i, col in enumerate(columns)}

def change_column_names(table_name, column_mapping, engine, to_english=True):
    """修改表的列名"""
    with engine.begin() as connection:
        for original_name, new_name in column_mapping.items():
            if to_english:
                if original_name == 'dt':
                    connection.execute(text(f"ALTER TABLE `{table_name}` CHANGE `{original_name}` `{new_name}` DATE;"))
                else:
                    connection.execute(text(f"ALTER TABLE `{table_name}` CHANGE `{original_name}` `{new_name}` VARCHAR(255);"))
            else:
                if original_name == 'dt':
                    connection.execute(text(f"ALTER TABLE `{table_name}` CHANGE `{new_name}` `{original_name}` DATE;"))
                else:
                    connection.execute(text(f"ALTER TABLE `{table_name}` CHANGE `{new_name}` `{original_name}` VARCHAR(255);"))

def sanitize_filename(filename):
    """清理文件名，使其符合Windows文件命名规范"""
    sanitized = re.sub(r'[<>:"/\\|?*]', '_', filename)
    return sanitized

print("数据清洗函数定义完成")

### 4.2 数据转换

In [ ]:
def pro_data(wsd_data, table_name, engine):
    """处理数据并插入数据库"""
    # 将 dt 列转换为 datetime 对象
    wsd_data['dt'] = pd.to_datetime(wsd_data['dt'], errors='coerce')
    wsd_data = wsd_data.dropna(subset=['dt'])
    wsd_data['dt'] = wsd_data['dt'].dt.strftime('%Y-%m-%d')
    
    # 将 NaN 值替换为 None
    wsd_data = wsd_data.replace({pd.NA: None, pd.NaT: None, float('nan'): None})
    
    if wsd_data.empty:
        print("No valid data to insert.")
        return

    # 生成列名映射
    column_mapping = generate_column_mapping(wsd_data.columns)
    wsd_data = wsd_data.rename(columns=column_mapping)

    inspector = inspect(engine)
    table_exists = inspector.has_table(table_name)
    
    if table_exists:
        existing_columns = inspector.get_columns(table_name)
        existing_columns_names = [col['name'] for col in existing_columns]
        wsd_data_columns = wsd_data.columns.tolist()

        # 检查并添加缺失的列
        for col in wsd_data_columns:
            if col not in existing_columns_names:
                col_type = "FLOAT" if pd.api.types.is_numeric_dtype(wsd_data[col]) else "VARCHAR(255)"
                with engine.begin() as connection:
                    connection.execute(text(f"ALTER TABLE `{table_name}` ADD COLUMN `{col}` {col_type};"))

        insert_columns = ', '.join([f"`{col}`" for col in wsd_data_columns])
        update_columns = ', '.join([f"`{col}` = VALUES(`{col}`)" for col in wsd_data_columns])
        value_placeholders = ', '.join([f":{col}" for col in wsd_data_columns])

        insert_query = text(f"""
        INSERT INTO `{table_name}` ({insert_columns})
        VALUES ({value_placeholders})
        ON DUPLICATE KEY UPDATE {update_columns};
        """)

        # 插入或更新数据
        with engine.begin() as connection:
            for _, row in wsd_data.iterrows():
                try:
                    params = {col: row[col] for col in wsd_data_columns}
                    connection.execute(insert_query, params)
                except Exception as e:
                    print(f"Error inserting row: {row}")
                    print(f"Exception: {e}")
        print('更新完成')

print("数据转换函数定义完成")

## 5. 核心逻辑

### 5.1 主函数实现

In [ ]:
def fetch_investor_data():
    """
    获取各交易所投资者数据
    包含：上交所、深交所、中债、上清所
    """
    now = datetime.now()
    year = now.year
    month = now.month
    
    results = {
        'sse': [],  # 上交所
        'szse': [], # 深交所
        'chinabond': [], # 中债
        'shclearing': []  # 上清所
    }
    
    for month1 in range(month-2, month):
        year1 = year
        if month1 == -1:
            month1 = 11
            year1 = year - 1
        elif month1 == 0:
            month1 = 12
            year1 = year - 1
        
        trade_date = f'{year1}-{month1}'
        print(f"处理日期: {trade_date}")
        
        # 上交所数据
        sse_df = fetch_sse_data(year1, month1)
        if sse_df is not None:
            results['sse'].append(sse_df)
    
    return results

def calculate_bond_index():
    """
    计算债券指数相关数据
    """
    sql = """
    SELECT 
        distinct
        case when
        (A.classify2 LIKE '%%中债%%' AND A.classify2 LIKE '%%城投债%%') then '是'
        else '否' end AS 是否城投,
        CASE
            WHEN classify2 LIKE '%%非公开%%' THEN '私募'
            ELSE '公募'
        END AS 发行方式,
        CASE
            WHEN classify2 LIKE '%%可续期%%' THEN '是'
            ELSE '否'
        END AS 是否永续,
        CASE WHEN classify2 LIKE '%%次级可续期%%' THEN '是' ELSE '否' END AS 是否次级,
        SUBSTRING(
            REPLACE(classify2, '＋', '+'),
            LOCATE('(', REPLACE(classify2, '＋', '+')) + 1,
            CHAR_LENGTH(classify2) - LOCATE('(',
                        REPLACE(classify2, '＋', '+')) - 1
        ) AS 隐含评级,
        t.term as 曲线期限
        FROM bond.basicinfo_curve A
        CROSS JOIN yq.目标期限 t
        WHERE (A.classify2 LIKE '%%中债%%' AND A.classify2 LIKE '%%城投债%%') 
        or (A.classify2 LIKE '%%中债%%' AND (A.classify2 LIKE '%%产业%%' or A.classify2 LIKE '%%企业%%'))
    """
    
    try:
        with sql_engine.begin() as conn:
            df = pd.read_sql(sql, conn)
            df.to_sql('曲线全样本', conn, if_exists='replace', index=False)
            print("债券指数计算完成")
            return df
    except Exception as e:
        print(f"债券指数计算失败: {e}")
        return None

print("核心逻辑函数定义完成")

### 5.2 辅助函数

In [ ]:
def execute_sql_with_retry(query, params=None, max_retries=3, engine=None):
    """执行SQL（带重试）"""
    if engine is None:
        engine = sql_engine
        
    for attempt in range(max_retries):
        try:
            with engine.begin() as connection:
                if params:
                    connection.execute(text(query), params)
                else:
                    connection.execute(text(query))
            return True
        except Exception as e:
            print(f"数据库错误 (尝试 {attempt + 1}): {e}")
            if attempt < max_retries - 1:
                sleep(5)
            else:
                raise
    return False

def fetch_with_retry(url, max_retries=3, delay=60, headers=None):
    """获取数据（带限流重试）"""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            return response
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429:  # Too Many Requests
                wait_time = delay * (attempt + 1)
                print(f"API限流，等待 {wait_time} 秒...")
                sleep(wait_time)
            else:
                raise
    return None

print("辅助函数定义完成")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def run_daily_tasks():
    """
    执行每日脚本任务
    """
    print(f"="*50)
    print(f"开始执行每日脚本任务 - {datetime.now()}")
    print(f"="*50)
    
    tasks_results = {}
    
    # 任务1: 获取投资者数据
    print("\n[任务1] 获取投资者数据...")
    try:
        investor_data = fetch_investor_data()
        tasks_results['investor_data'] = 'success'
        print("投资者数据获取完成")
    except Exception as e:
        tasks_results['investor_data'] = f'failed: {e}'
        print(f"投资者数据获取失败: {e}")
    
    # 任务2: 计算债券指数
    print("\n[任务2] 计算债券指数...")
    try:
        bond_index = calculate_bond_index()
        tasks_results['bond_index'] = 'success' if bond_index is not None else 'failed'
        print("债券指数计算完成")
    except Exception as e:
        tasks_results['bond_index'] = f'failed: {e}'
        print(f"债券指数计算失败: {e}")
    
    print(f"\n" + "="*50)
    print(f"每日脚本任务执行完成 - {datetime.now()}")
    print(f"="*50)
    
    return tasks_results

# 执行任务
results = run_daily_tasks()
print("\n任务执行结果:")
for task, status in results.items():
    print(f"  - {task}: {status}")

### 6.2 结果验证

In [ ]:
def verify_data():
    """
    验证数据更新情况
    """
    verification_results = {}
    
    # 检查投资者数据表
    tables_to_check = [
        'yq.investor_sh',
        'yq.investor_sz', 
        'yq.investor_zz',
        'yq.investor_sq',
        'yq.曲线全样本'
    ]
    
    for table in tables_to_check:
        try:
            sql = f"SELECT COUNT(*) as cnt FROM {table}"
            with sql_engine.begin() as conn:
                result = conn.execute(text(sql)).fetchone()
                verification_results[table] = result[0]
                print(f"{table}: {result[0]} 条记录")
        except Exception as e:
            verification_results[table] = f"Error: {e}"
            print(f"{table}: 检查失败 - {e}")
    
    return verification_results

# 执行验证
print("数据验证结果:")
verify_data()

## 7. 工具函数（可复用）

In [ ]:
# ============================================
# 数据库工具函数
# ============================================

def get_table_columns(table_name, engine=None):
    """获取表的所有列名"""
    if engine is None:
        engine = sql_engine
    inspector = inspect(engine)
    columns = inspector.get_columns(table_name)
    return [col['name'] for col in columns]

def table_exists(table_name, engine=None):
    """检查表是否存在"""
    if engine is None:
        engine = sql_engine
    inspector = inspect(engine)
    return inspector.has_table(table_name)

def insert_data_to_db(df, table_name, engine=None, if_exists='append'):
    """将DataFrame插入数据库"""
    if engine is None:
        engine = sql_engine
    try:
        with engine.begin() as conn:
            df.to_sql(table_name, con=conn, if_exists=if_exists, index=False)
        return True
    except Exception as e:
        print(f"插入数据失败: {e}")
        return False

# ============================================
# 日期工具函数
# ============================================

def get_date_range(start_date, end_date):
    """获取日期范围列表"""
    return pd.date_range(start=start_date, end=end_date, freq='D')

def get_previous_trading_days(n=5):
    """获取前n个交易日（简化版，不考虑节假日）"""
    dates = []
    current = datetime.now()
    while len(dates) < n:
        current -= timedelta(days=1)
        if current.weekday() < 5:  # 周一到周五
            dates.append(current.strftime('%Y-%m-%d'))
    return dates

# ============================================
# 日志工具函数
# ============================================

def log_task_status(task_name, status, message=''):
    """记录任务状态"""
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    log_entry = f"[{timestamp}] {task_name}: {status}"
    if message:
        log_entry += f" - {message}"
    print(log_entry)
    return log_entry

print("工具函数加载完成")

---

## 脚本清单

| 脚本名称 | 功能描述 | 执行时间 | 数据源 |
|---------|----------|---------|--------|
| td.py | 同花顺数据获取 | 1:00 | THS API |
| thslc.py | THS理财数据获取 | 9:00, 19:30 | THS API |
| tzzxw.py | 投资者资讯数据 | 19:30, 23:30 | 交易所API |
| xyzgm.py | 公告数据获取 | 2:00, 20:00 | THS API |
| yyxt.py | 研报数据获取 | 6:00 | THS API |
| zhaishiguimo.py | 实时资讯获取 | 6:00, 20:00, 21:00 | THS API |
| zqdq.py | 定向数据获取 | 20:30, 21:00 | THS API |
| cind.py | 行业数据计算 | 按需 | 内部计算 |
| cyhs.py | 宏观数据处理 | 3:30 | 内部计算 |
| jrzyq.py | 金融市场数据 | 23:30, 24:00 | 交易所API |
| qxqyb.py | 债券指数计算 | 6:00 | 内部计算 |
| yhzf.py | 宏观趋势分析 | 18:00, 19:00 | 内部计算 |
| yjfxct.py | 衍生品分析 | 17:00, 18:30, 20:00, 21:00 | 内部计算 |
| yuqing.py | 情绪分析 | 5:30, 20:00, 21:00, 21:30 | 第三方API |
| yycjqb.py | 行为分析 | 22:00-23:00 | 内部计算 |
| licaishouyi.py | 资金流向 | 9:00, 15:00, 19:00, 20:00, 21:00 | 内部计算 |
| zqfx.py | 资金分析 | 20:30, 21:00 | 内部计算 |

---

**最后更新**: 2026-02-14